In [ ]:
import os
import mne
import mne_bids
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from glob import glob
from mne_bids import BIDSPath, read_raw_bids
from scipy.stats import ttest_rel
from numpy import mean, sqrt, square
import matplotlib.patches as mpatches
from scipy.io import wavfile
from scipy import signal
from PIL import Image
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks
from statsmodels.stats.multitest import multipletests
plt.ioff()
# %matplotlib widget

### define inputs

In [ ]:
bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"

# outputs
# deriv_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids\derivatives\epochs-python-stimtrack-unfiltered"
deriv_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids\derivatives\epochs-python-stimtrack"
if not os.path.exists(deriv_dir):
    os.makedirs(deriv_dir)
    
events_dir = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\stimtrack-compare\python_v2"

# Preprocess data (BIDS)

In [ ]:
task_list = ['active', 'passive']
passive_hex = '#4477AA'
active_hex = '#CC6677'

In [ ]:

# reference and filter EEG data, set events from stimtrack
for sub_num in range(21, 35):
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, 
                                 datatype='eeg', 
                                 subject=str(sub_label), 
                                 task=task_label, 
                                 run=run_label)
            
            try:
                data = read_raw_bids(bids_path, verbose=False)
                data.load_data()

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # filter data
                data_filtered = data_ref.copy().filter(l_freq=65, h_freq=2000, verbose=True)

                # loading events from the stim-track-generated events
                events_fpath = os.path.join(events_dir, f'sub-{sub_label}', f'sub-{sub_label}_task-{task_label}_run-{run_label}_stimtrack_events.tsv')
                events_df = pd.read_csv(events_fpath, delimiter='\t') 
                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_ref, 
                                    events=events_from_annot, 
                                    event_id=event_dict2,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-0.04, tmax=0.4, 
                                    baseline=[-0.04, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}")
                print(f"Error: {e}")
        
        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
        except Exception as e:
            print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}')
            print(f'Error: {e}')

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_run-all_event-stimtrack_epochs.fif'
            all_epochs.save(os.path.join(deriv_dir, out_base), overwrite=True)
        except:
            continue


# Analyze data

In [ ]:
act_pos_avgs = []
act_neg_avgs = []
act_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-active_run-all_event-stimtrack_epochs.fif') 
                  for sub_num in range(2,35))]
for act_ep in act_eps[0]:
    try:
        sub_active_epochs = mne.read_epochs(act_ep[0], verbose=False)
        act_pos_avgs.append(sub_active_epochs['1'].average())
        act_neg_avgs.append(sub_active_epochs['2'].average())
    except:
        pass

pas_pos_avgs = []
pas_neg_avgs = []
pas_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-passive_run-all_event-stimtrack_epochs.fif') 
                  for sub_num in range(2,35))]
for pas_ep in pas_eps[0]:
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep[0], verbose=False)
        pas_pos_avgs.append(sub_passive_epochs['1'].average())
        pas_neg_avgs.append(sub_passive_epochs['2'].average())
    except:
        pass


In [ ]:
passive_avgs = [mne.combine_evoked([pas_pos_avgs[x], pas_neg_avgs[x]], weights='nave')
                for x in range(len(pas_pos_avgs))]
active_avgs = [mne.combine_evoked([act_pos_avgs[x], act_neg_avgs[x]], weights='nave')
                for x in range(len(act_pos_avgs))]

In [ ]:
# create grand averages per polarity per condition
pas_pos_grandavg = mne.grand_average(pas_pos_avgs)
pas_neg_grandavg = mne.grand_average(pas_neg_avgs)
act_pos_grandavg = mne.grand_average(act_pos_avgs)
act_neg_grandavg = mne.grand_average(act_neg_avgs)

# create grand averages per condition
passive_grandavg = mne.grand_average(pas_pos_avgs+pas_neg_avgs)
active_grandavg = mne.grand_average(act_pos_avgs+act_neg_avgs)

In [ ]:
evokeds = dict(passive=passive_grandavg, active=active_grandavg)
evokeds_cropped = dict(passive=passive_grandavg.crop(-0.04, 0.3), 
                       active=active_grandavg.crop(-0.04, 0.3))

In [ ]:
# figs = mne.viz.plot_compare_evokeds(evokeds, 
#                              title='Grand average FFR',
#                              truncate_xaxis=False)
# ax = figs[0].axes[0]
# ax.set_xlabel('Time (s)')
# ax.set_ylabel('Amplitude (uV)')
# figs[0].savefig('grand_average_ffr_by_condition.svg')

In [ ]:
# Power spectra
passive_psd = passive_grandavg.compute_psd(method='welch',
                                           tmin=0.1, tmax=0.2, fmin=50, fmax=700)
active_psd = active_grandavg.compute_psd(method='welch',
                                           tmin=0.1, tmax=0.2, fmin=50, fmax=700)

In [ ]:
# Spectrograms
frequencies = np.arange(65,550)

passive_power = passive_grandavg.compute_tfr("morlet", freqs=frequencies)
active_power = active_grandavg.compute_tfr("morlet", freqs=frequencies)

### Power stats

In [ ]:
def compute_power(avg_evoked, f_low=90, f_high=110, t_low=0.1, t_high=0.2):
    # Compute the spectrogram (time-frequency representation)
    power = mne.time_frequency.tfr_multitaper(avg_evoked, 
                                            freqs=np.arange(f_low, f_high, 1), 
                                            n_cycles=2, time_bandwidth=4.0,
                                            average=True, return_itc=False)

    # Select the time and frequency bands
    time_mask = (power.times >= t_low) & (power.times <= t_high)
    freq_mask = (power.freqs >= f_low) & (power.freqs <= f_high)

    # Extract the power within the specified bands
    selected_power = power.data[:, freq_mask, :][:, :, time_mask]

    # Average power over the selected time and frequency bands
    average_power = selected_power.mean(axis=2).mean(axis=1)[0]
    return average_power

In [ ]:
f_low=90
f_high=110
t_low=0.1
t_high=0.2

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

### RMS SNR stats

In [ ]:
# helper function to compute RMS SNR
def compute_rms(evoked):
    baseline_ind_bounds = evoked.time_as_index(evoked.baseline)
    response_ind_bounds = evoked.time_as_index([0.100, 0.200])

    evoked_baseline = evoked.data[0,baseline_ind_bounds[0]:baseline_ind_bounds[1]]
    evoked_response = evoked.data[0,response_ind_bounds[0]:response_ind_bounds[1]]
    rms_baseline = sqrt(mean(square(evoked_baseline)))
    rms_response = sqrt(mean(square(evoked_response)))

    rms_snr = rms_response / rms_baseline
    return rms_snr

In [ ]:
# compute RMS SNR for each condition for the grand average
task_evoked_dict = {}
task_evoked_dict['passive'] = passive_grandavg
task_evoked_dict['active']  = active_grandavg

for sx, stim in enumerate(task_evoked_dict):
    s_evoked = task_evoked_dict[stim]
    rms_snr = compute_rms(s_evoked)

    print(f'{stim} RMS SNR: {rms_snr:.04f}')


In [ ]:
# compute RMS SNR for each condition
pas_rms_list = []
for evk in pas_pos_avgs+pas_neg_avgs:
    rms_snr = compute_rms(evk)
    pas_rms_list.append(rms_snr)

act_rms_list = []
for evk in act_pos_avgs+act_neg_avgs:
    rms_snr = compute_rms(evk)
    act_rms_list.append(rms_snr)

# Create a DataFrame for RMS SNR values
rms_dict = {'passive': pas_rms_list,  
              'active': act_rms_list}
rms_df = pd.DataFrame(rms_dict)


In [ ]:
# calculate mean RMS SNR for each condition
rms_pass_mean = rms_df['passive'].mean()
rms_act_mean = rms_df['active'].mean()
print(f'Mean passive RMS ratio: {rms_pass_mean:.03f}')
print(f'Mean active RMS ratio: {rms_act_mean:.03f}')

# Perform paired t-test
t_stat, p_value = ttest_rel(rms_dict['passive'], rms_dict['active'])
print(f"RMS ratio t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

### Autocorrelation/Pitch tracking

In [ ]:
def compute_pitch_and_conf(evoked,
                           win_dur=0.040, hop_dur=0.010,
                           fmin=60, fmax=200,
                           strength_thresh=0.15, smooth_k=3):
    signal = evoked.data[0]
    times = evoked.times
    sfreq = evoked.info['sfreq'] if hasattr(evoked, 'info') else 1.0 / np.diff(times)[0]

    win_samps = max(3, int(round(win_dur * sfreq)))
    hop_samps = max(1, int(round(hop_dur * sfreq)))
    lag_min = max(1, int(round(sfreq / fmax)))
    lag_max = int(round(sfreq / fmin))

    pitch_times = []
    pitch_hz = []
    peak_strength = []

    # compute pitch (autocorr) per window
    for start in range(0, len(signal) - win_samps + 1, hop_samps):
        w = signal[start:start + win_samps].copy()
        w = w - np.mean(w)
        ac_full = np.correlate(w, w, mode='full')
        ac = ac_full[ac_full.size // 2:]  # positive lags
        if ac.size <= lag_min:
            pitch = np.nan
            strength = 0.0
        else:
            norm_ac = ac / (ac[0] if ac[0] != 0 else 1.0)
            search = norm_ac[lag_min: min(lag_max + 1, len(norm_ac))]
            if search.size == 0:
                pitch = np.nan
                strength = 0.0
            else:
                idx = int(np.argmax(search))
                strength = float(search[idx])
                lag = idx + lag_min
                pitch = float(sfreq / lag) if strength >= strength_thresh else np.nan

        center_time = times[start + win_samps // 2]
        pitch_times.append(center_time)
        pitch_hz.append(pitch)
        peak_strength.append(strength)

    pitch_times = np.array(pitch_times)
    pitch_hz = np.array(pitch_hz)
    peak_strength = np.array(peak_strength)

    # smoothing that ignores NaNs
    def moving_avg_ignore_nan(x, k=3):
        mask = ~np.isnan(x)
        x0 = np.where(mask, x, 0.0)
        num = np.convolve(x0, np.ones(k), mode='same')
        den = np.convolve(mask.astype(float), np.ones(k), mode='same')
        den[den == 0] = np.nan
        return num / den

    pitch_hz_smooth = moving_avg_ignore_nan(pitch_hz, k=smooth_k)

    # compute per-window confidence metrics (rmax, PNR, z)
    rmax_list, pnr_list, z_list, valid_times = [], [], [], []
    for start in range(0, len(signal) - win_samps + 1, hop_samps):
        w = signal[start:start + win_samps].copy()
        w = w - np.mean(w)
        ac_full = np.correlate(w, w, mode='full')
        ac = ac_full[ac_full.size // 2:]
        if ac.size <= lag_min or lag_min >= len(ac):
            rmax_list.append(np.nan); pnr_list.append(np.nan); z_list.append(np.nan)
            valid_times.append(times[start + win_samps // 2])
            continue

        norm_ac = ac / (ac[0] if ac[0] != 0 else 1.0)
        search = norm_ac[lag_min: min(lag_max + 1, len(norm_ac))]
        if search.size == 0:
            rmax_list.append(np.nan); pnr_list.append(np.nan); z_list.append(np.nan)
            valid_times.append(times[start + win_samps // 2])
            continue

        idx_rel = int(np.argmax(search))
        rmax = float(search[idx_rel])

        peaks, _ = find_peaks(search)
        if peaks.size == 0:
            sorted_idx = np.argsort(search)
            next_max = float(search[sorted_idx[-2]]) if sorted_idx.size >= 2 else 0.0
        else:
            peak_vals = search[peaks]
            mask = peaks != idx_rel
            other_vals = peak_vals[mask]
            next_max = float(other_vals.max()) if other_vals.size > 0 else 0.0

        pnr = (rmax / (next_max + 1e-12)) if next_max > 0 else np.inf
        search_mean = float(np.mean(search))
        search_std = float(np.std(search)) if float(np.std(search)) > 0 else 1e-12
        z = (rmax - search_mean) / search_std

        rmax_list.append(rmax)
        pnr_list.append(pnr)
        z_list.append(z)
        valid_times.append(times[start + win_samps // 2])

    rmax_arr = np.array(rmax_list)
    pnr_arr = np.array(pnr_list)
    z_arr = np.array(z_list)
    valid_times = np.array(valid_times)

    return {
        'times': pitch_times,
        'pitch_hz': pitch_hz,
        'pitch_hz_smooth': pitch_hz_smooth,
        'peak_strength': peak_strength,
        'conf_rmax': rmax_arr,
        'conf_pnr': pnr_arr,
        'conf_z': z_arr,
        'conf_times': valid_times,
    }

def _conf_to_halfwidth(conf_rmax, max_band_hz=8.0, min_band_hz=1.0):
    hw = min_band_hz + (max_band_hz - min_band_hz) * (1.0 - np.clip(conf_rmax, 0, 1))
    hw[np.isnan(conf_rmax)] = np.nan
    return hw

def _align_conf_to_pitch(pitch_times, conf_times, conf_values):
    return np.interp(pitch_times, conf_times, conf_values, left=np.nan, right=np.nan)

In [ ]:
# compute pitch for both conditions
pitch_pass = compute_pitch_and_conf(passive_grandavg.crop(-0.04, 0.3))
pitch_act  = compute_pitch_and_conf(active_grandavg.crop(-0.04, 0.3))
# pitch_stim = compute_pitch_and_conf(stim_evoked)

# Visualization: spectrograms with pitch overlays and confidence traces
frequencies = np.arange(65, 250)
passive_power = passive_grandavg.compute_tfr("morlet", freqs=frequencies)
active_power  = active_grandavg.compute_tfr("morlet", freqs=frequencies)

# choose confidence metric to color markers ('conf_rmax', 'conf_pnr', 'conf_z')
conf_key = 'conf_rmax'

#conf for passive
conf_p = _align_conf_to_pitch(pitch_pass['times'], pitch_pass['conf_times'], pitch_pass[conf_key])
hw_p = _conf_to_halfwidth(conf_p)
upper_p = pitch_pass['pitch_hz_smooth'] + hw_p
lower_p = pitch_pass['pitch_hz_smooth'] - hw_p

# conf for active
conf_a = _align_conf_to_pitch(pitch_act['times'], pitch_act['conf_times'], pitch_act[conf_key])
hw_a = _conf_to_halfwidth(conf_a)
upper_a = pitch_act['pitch_hz_smooth'] + hw_a
lower_a = pitch_act['pitch_hz_smooth'] - hw_a

In [ ]:
# Pitch tracking statistics
# Ensure passive_avgs and active_avgs exist and have matching subjects
n_pass = len(passive_avgs)
n_act = len(active_avgs)
n_sub = min(n_pass, n_act)
if n_sub == 0:
    raise RuntimeError("No subjects available in passive_avgs or active_avgs.")

# Compute per-subject pitch/conf if not already computed per subject
pass_pitch_list = [compute_pitch_and_conf(passive_avgs[i]) for i in range(n_sub)]
act_pitch_list  = [compute_pitch_and_conf(active_avgs[i])  for i in range(n_sub)]

# Choose a common time base (use the first subject's conf_times)
times_common = pass_pitch_list[0]['conf_times']

# Interpolate rmax to the common time base and build arrays (subjects x time)
def stack_rmax(pitch_list, times_target):
    arr = []
    for p in pitch_list:
        t = p['conf_times']
        r = p['conf_rmax']
        # handle NaNs and possible differing lengths via interpolation
        r_interp = np.interp(times_target, t, np.where(np.isfinite(r), r, np.nan),
                             left=np.nan, right=np.nan)
        # preserve NaNs where original had no valid support by masking with a nearest valid check
        # (interp fills with nearest; convert extreme extrapolated values back to NaN where outside original range)
        outside = (times_target < t.min()) | (times_target > t.max())
        r_interp[outside] = np.nan
        arr.append(r_interp)
    return np.vstack(arr)

rmax_pass = stack_rmax(pass_pitch_list, times_common)  # shape (n_sub, n_time)
rmax_act  = stack_rmax(act_pitch_list,  times_common)

# Compute means and SEM across subjects
mean_pass = np.nanmean(rmax_pass, axis=0)
mean_act  = np.nanmean(rmax_act, axis=0)
sem_pass  = np.nanstd(rmax_pass, axis=0, ddof=1) / np.sqrt(np.sum(np.isfinite(rmax_pass), axis=0))
sem_act   = np.nanstd(rmax_act, axis=0, ddof=1)  / np.sqrt(np.sum(np.isfinite(rmax_act), axis=0))

# Paired t-test at each timepoint (omit NaNs)
p_vals = np.full(times_common.shape, np.nan)
t_stats = np.full(times_common.shape, np.nan)
for ti in range(len(times_common)):
    a = rmax_act[:, ti]
    b = rmax_pass[:, ti]
    # require at least 2 paired non-nan samples
    mask = np.isfinite(a) & np.isfinite(b)
    if np.sum(mask) >= 2:
        t, p = ttest_rel(a[mask], b[mask], nan_policy='omit')
        t_stats[ti] = t
        p_vals[ti] = p

# Multiple comparisons correction (FDR)
valid = np.isfinite(p_vals)
reject = np.zeros_like(p_vals, dtype=bool)
if np.any(valid):
    rej, pvals_corr, _, _ = multipletests(p_vals[valid], alpha=0.05, method='fdr_bh')
    reject[valid] = rej
else:
    pvals_corr = np.array([])

# Print summary statistics
n_sig = int(np.sum(reject))
print(f"Subjects used (paired): {n_sub}")
print(f"Number of timepoints with significant difference (FDR q<0.05): {n_sig}")
if n_sig > 0:
    print("Significant time ranges (s):")
    # group contiguous significant samples into ranges
    sig_idx = np.where(reject)[0]
    ranges = []
    start = sig_idx[0]
    prev = sig_idx[0]
    for idx in sig_idx[1:]:
        if idx == prev + 1:
            prev = idx
            continue
        else:
            ranges.append((times_common[start], times_common[prev]))
            start = idx
            prev = idx
    ranges.append((times_common[start], times_common[prev]))
    for r in ranges:
        print(f"  {r[0]:.3f} - {r[1]:.3f} s")

### Read in stimulus

In [ ]:
# read in stimulus waveform
stim_dir = os.path.abspath(r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids\stimuli")
stim_pos_fpath = os.path.join(stim_dir, 'Da_Stimulus_44100Hz_pol1.wav')
stim_neg_fpath = os.path.join(stim_dir, 'Da_Stimulus_44100Hz_pol2.wav')
sr, stim_pos = wavfile.read(stim_pos_fpath)
sr, stim_neg = wavfile.read(stim_neg_fpath)
stim_duration = len(stim_pos) / sr
stim_time = np.linspace(0, stim_duration, num=len(stim_pos))

stim_pos = stim_pos.astype(float) / np.max(np.abs(stim_pos))  
stim_neg = stim_neg.astype(float) / np.max(np.abs(stim_neg))

stim_pos = stim_pos.astype(np.float32) / np.iinfo(np.int16).max
stim_neg = stim_neg.astype(np.float32) / np.iinfo(np.int16).max


### Plotting all results (Figure 1 - 2 in manuscript)

In [ ]:
# CURRENT figure 1: montage and stimulus
plt.close('all')
fig, axd = plt.subplot_mosaic(
    [['stimulus', 'montage']],
     figsize=(12, 6),
     constrained_layout=True
)

# montage image - image is hand-drawn, then put through Claude to smooth out hand-drawn lines
montage_img = Image.open(r"C:\Users\Laura\Documents\GitHub\EAM1-FFR\montage_figure.png")
axd['montage'].imshow(montage_img)
axd['montage'].axis('off')

# stimulus waveform
axd['stimulus'].remove()
gs_sub = gridspec.GridSpecFromSubplotSpec(
    2, 1,
    subplot_spec=axd['stimulus'].get_subplotspec(),
    hspace=0.05
)

ax_stim_pos = fig.add_subplot(gs_sub[0, 0])
ax_stim_neg = fig.add_subplot(gs_sub[1, 0], 
                              sharex=ax_stim_pos, 
                              sharey=ax_stim_pos)
ax_stim_pos.plot(stim_time,
                 stim_pos,
                 color='#009E73',
                 linewidth=1)
ax_stim_neg.plot(stim_time,
                 stim_neg,
                 color='#E69F00',
                 linewidth=1)

# ax_stim_pos.set_xlabel('Time (ms)')
ax_stim_pos.set_xlim(0, stim_duration)
ax_stim_pos.set_yticks([])
ax_stim_pos.spines['top'].set_visible(False)
ax_stim_pos.spines['right'].set_visible(False)
ax_stim_pos.spines['left'].set_visible(False)
ax_stim_pos.tick_params(direction="in")
ax_stim_pos.set_title('/da/ Positive Polarity')

ax_stim_neg.set_xlabel('Time (ms)')
ax_stim_neg.set_xlim(0, stim_duration)
ax_stim_neg.set_yticks([])
ax_stim_neg.spines['top'].set_visible(False)
ax_stim_neg.spines['right'].set_visible(False)
ax_stim_neg.spines['left'].set_visible(False)
ax_stim_neg.tick_params(direction="in")
ax_stim_neg.set_title('/da/ Negative Polarity')

ax_stim_neg.set_xticks([0, .05, .10, .15])

fig.savefig('figure1_v5.svg', dpi=600)

In [ ]:
# CURRENT figure 2: ffr, rms and power box plots, pitch tracking line plot, spectrograms, spectra

fig, axd = plt.subplot_mosaic(
    [["waveform", "waveform", "spectra", "spectra"],
     ["passive_spect", "passive_spect", "active_spect", "active_spect"],
     ["pitch", "pitch", "power", "rms"]],
    constrained_layout=True,
     figsize=(9, 9)
)

# set figure legend
fig.get_layout_engine().set(rect=[0, 0, 1, 0.92], hspace=0.15, wspace=0.08)
active_patch  = mpatches.Patch(color=active_hex,  label='Active')
passive_patch = mpatches.Patch(color=passive_hex, label='Passive')

fig.legend(
    handles=[passive_patch, active_patch],
    loc='upper right',
    ncols=2,
    frameon=False,
    fontsize=10,
)

mne.viz.plot_compare_evokeds(evokeds_cropped,
                             truncate_xaxis=False, 
                             axes=axd['waveform'],
                             colors={'passive': passive_hex, 'active': active_hex},
                             legend=False)

n_times = len(list(evokeds_cropped.values())[0].times)
for line in axd['waveform'].get_lines():
    line.set_linewidth(1)
    print(len(line.get_xdata()), line.get_color, line.get_linestyle())
axd['waveform'].set_ylabel('Amplitude (µV)')
axd['waveform'].tick_params(direction="in")

# PSDs
passive_psd.plot(axes=axd["spectra"], 
                 amplitude=False, 
                 average=True, 
                 show=False,
                 color=passive_hex)

active_psd.plot(axes=axd["spectra"], 
                amplitude=False, 
                average=True, 
                show=False, 
                color=active_hex)
for line in axd["spectra"].get_lines():
    line.set_linewidth(1.5)
axd["spectra"].set_ylim([-100, -20])

axd["spectra"].set_ylabel('Power (dB)')
axd["spectra"].set_xlabel('Frequency (Hz)')
axd["spectra"].set_xticks([100, 200, 300, 400, 500, 600])
axd["spectra"].set_xticklabels([100, '', 300, '', 500, ''])
axd["spectra"].set_xticks([100, 200, 300, 400, 500, 600])
axd["spectra"].set_xticklabels([100, '', 300, '', 500, ''])
axd["spectra"].tick_params(direction='in')

sns.swarmplot(power_df, 
              edgecolor='k', 
              linewidth=0.5, 
              log_scale=True, 
              ax=axd['power'],
              size=2,
              palette=[passive_hex, active_hex])
sns.boxplot(power_df, 
            linecolor='k',
            fliersize=0, 
            ax=axd['power'], 
            palette=[passive_hex, active_hex])

axd['power'].set_yscale('log')
axd['power'].set_ylabel(
    f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)'
)
axd['power'].set_xlabel('')
axd['power'].set_xticklabels([])
axd['power'].set_xlabel('')
axd['power'].set_xticks([])
axd['power'].tick_params(direction='in')


sns.boxplot(
    data=rms_df,
    ax=axd['rms'],
    linewidth=1,
    fliersize=0,
    linecolor='k',
    palette=[passive_hex, active_hex]
)

sns.swarmplot(
    data=rms_df,
    ax=axd['rms'],
    edgecolor='k',
    linewidth=0.5,
    size=2,
    palette=[passive_hex, active_hex]
)

# axd['rms'].set_title('RMS ratio')
axd['rms'].set_ylabel('RMS ratio (response / baseline)')
axd['rms'].set_xlabel('')
axd['rms'].set_xticklabels([])
axd['rms'].set_xlabel('')
axd['rms'].set_xticks([])
axd['rms'].tick_params(direction='in')


# pitch tracking
conf_thresh = 0.15  # same threshold you already use

axd['pitch'].axhline(100, color='k', ls='--', lw=0.8, alpha=0.5)
# ----- Passive FFR F0 -----
mask_pass = (
    ~np.isnan(pitch_pass['pitch_hz_smooth']) &
    (pitch_pass['conf_rmax'] >= conf_thresh)
)

axd['pitch'].plot(
    pitch_pass['times'],
    pitch_pass['pitch_hz_smooth'],
    color=passive_hex,
    lw=2.0,
)
axd['pitch'].fill_between(
    pitch_pass['times'],
    lower_p,
    upper_p,
    linewidth=0,
    alpha=0.3,
    color=passive_hex)

# ----- Active FFR F0 -----
mask_act = (
    ~np.isnan(pitch_act['pitch_hz_smooth']) &
    (pitch_act['conf_rmax'] >= conf_thresh)
)

axd['pitch'].plot(
    pitch_act['times'],
    pitch_act['pitch_hz_smooth'],
    color=active_hex,
    lw=2.0,
)
axd['pitch'].fill_between(
    pitch_act['times'],
    lower_a,
    upper_a,
    linewidth=0,
    alpha=0.3,
    color=active_hex)

#----- Formatting -----
# x_min, x_max = axd['waveform'].get_xlim()
# axd['pitch'].set_xlim(x_min, x_max)
axd['pitch'].set_ylim(60, 120)
axd['pitch'].set_xlabel('Time (s)')
axd['pitch'].set_ylabel('F0 (Hz)')
axd['pitch'].spines['top'].set_visible(False)
axd['pitch'].spines['right'].set_visible(False)
axd['pitch'].sharex(axd['waveform'])
axd['pitch'].tick_params(direction='in')

# ffr spectrograms
active_power.plot(axes=axd["active_spect"], 
                  tmax=0.3, 
                  vlim=[0, 5e-12],
                  colorbar=False,
                  show=False,
                  cmap='magma')

passive_power.plot(axes=axd["passive_spect"],
                   tmax=0.3, 
                   vlim=[0, 5e-12], 
                   colorbar=False, 
                   show=False,
                   cmap='magma')

axd['active_spect'].set_title('Active')
axd['passive_spect'].set_title('Passive')
axd['active_spect'].set_ylabel('Frequency (Hz)')
axd['passive_spect'].set_ylabel('')
axd['active_spect'].set_xlabel('Time (s)')
axd['passive_spect'].set_xlabel('Time (s)')
axd['active_spect'].tick_params(direction="in")
axd['passive_spect'].tick_params(direction="in")

im = axd['passive_spect'].images[0]  # grab the image from one of the axes
cbar = fig.colorbar(im, ax=[axd['passive_spect'], axd['active_spect']], 
                    location='bottom',  
                    shrink=0.3,
                    label='Power ($\\times 10^{-12}$)')

# final draw and save debug image to inspect output
# fig.canvas.draw()
out_debug = 'figure2_v6.svg'
fig.savefig(out_debug, dpi=600)

### alternative power analysis (not included in manuscript)

In [ ]:
tmin = 0.05
tmax = 0.15

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs:
    peak = evk.get_peak(tmin=tmin, tmax=tmax, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs:
    peak = evk.get_peak(tmin=0.05, tmax=0.15, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
              'active': act_peak_lat_list}
peak_df = pd.DataFrame(peak_lat_dict)

In [ ]:
passive_avgs[0].pick('Cz').get_peak()

In [ ]:
peak_df.head()

In [ ]:
# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_act_mean = peak_df['active'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} ms')
print(f'Mean active peak latency: {peak_act_mean:.03f} ms')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['active'])
print(f"peak t-statistic: {t_stat:.03f}\npeak p-value: {p_value:.05f}")

In [ ]:
# Plotting the power values
plt.figure(figsize=(5,4), dpi=200)
sns.swarmplot(peak_df, edgecolor='k', linewidth=1)
sns.boxplot(peak_df, fliersize=0)
#plt.ylim([0.0, 0.15])
plt.ylabel(f'Latency ({tmin}–{tmax} s)')
plt.title('FFR latency in passive vs active conditions')

plt.show()

## Individual polarities (not included in manuscript)

In [ ]:
evokeds_polar = dict(passive_positive=pas_pos_avgs, 
                     passive_negative=pas_neg_avgs, 
                     active_positive=act_pos_avgs, 
                     active_negative=act_neg_avgs)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,5), dpi=300)
mne.viz.plot_compare_evokeds(evokeds_polar, 
                             #[pas_pos_grandavg,pas_neg_grandavg,act_pos_grandavg, act_neg_grandavg],
                             ci=False,
                             truncate_xaxis=False,
                             legend='lower right',
                             title='Grand average FFR per condition and polarity',
                             axes=ax)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8), dpi=200)
ax = axes.flatten()
pas_pos_grandavg.plot(axes=ax[0], selectable=False, show=False)
pas_neg_grandavg.plot(axes=ax[1], selectable=False, show=False)
act_pos_grandavg.plot(axes=ax[2], selectable=False, show=False)
act_neg_grandavg.plot(axes=ax[3], selectable=False, show=False)

ax[0].set_title('task-passive\npolarity-positive')
ax[1].set_title('task-passive\npolarity-negative')
ax[2].set_title('task-active\npolarity-positive')
ax[3].set_title('task-active\npolarity-negative')

fig.suptitle(f'Grand average FFR per condition and polarity', size='x-large')


fig.tight_layout()

In [ ]:
# Power spectra
passive_pos_psd = pas_pos_grandavg.compute_psd(tmin=0.05, tmax=0.300, fmin=50, fmax=800)
passive_neg_psd = pas_neg_grandavg.compute_psd(tmin=0.05, tmax=0.300, fmin=50, fmax=800)
active_pos_psd = act_pos_grandavg.compute_psd(tmin=0.05, tmax=0.300, fmin=50, fmax=800)
active_neg_psd = act_neg_grandavg.compute_psd(tmin=0.05, tmax=0.300, fmin=50, fmax=800)

fig, axes = plt.subplots(2, 2, figsize=(8,6), dpi=200)
ax = axes.flatten()
passive_pos_psd.plot(axes=ax[0], amplitude=False, average=True, show=False)
passive_neg_psd.plot(axes=ax[1], amplitude=False, average=True, show=False)
active_pos_psd.plot(axes=ax[2], amplitude=False, average=True, show=False)
active_neg_psd.plot(axes=ax[3], amplitude=False, average=True, show=False)

ax[0].set_title('task-passive polarity-positive')
ax[1].set_title('task-passive polarity-negative')
ax[2].set_title('task-active polarity-positive')
ax[3].set_title('task-active polarity-negative')

fig.suptitle(f'Grand average FFR power spectra', size='x-large')

fig.tight_layout()
plt.show()
#plt.savefig(f'sub-{sub_label}_polarity-single_spectra.svg')

In [ ]:
frequencies = np.arange(80,120)

# Spectrograms
passive_pos_power = pas_pos_grandavg.compute_tfr("morlet", freqs=frequencies)
passive_neg_power = pas_neg_grandavg.compute_tfr("morlet", freqs=frequencies)
active_pos_power  = act_pos_grandavg.compute_tfr("morlet", freqs=frequencies)
active_neg_power  = act_neg_grandavg.compute_tfr("morlet", freqs=frequencies)

fig, axes = plt.subplots(2, 2, figsize=(8,6), dpi=200)
ax = axes.flatten()
passive_pos_power.plot(axes=ax[0], tmax=0.3, vlim=[0, 5e-12], show=False)
passive_neg_power.plot(axes=ax[1], tmax=0.3, vlim=[0, 5e-12], show=False)
active_pos_power.plot(axes=ax[2], tmax=0.3, vlim=[0, 5e-12], show=False)
active_neg_power.plot(axes=ax[3], tmax=0.3, vlim=[0, 5e-12], show=False)

ax[0].set_title('task-passive\npolarity-positive')
ax[1].set_title('task-passive\npolarity-negative')
ax[2].set_title('task-active\npolarity-positive')
ax[3].set_title('task-active\npolarity-negative')

fig.suptitle(f'Grand average FFR spectrograms',size='x-large')

fig.tight_layout()
plt.show()
#plt.savefig(f'sub-{sub_label}_polarity-single_spectra.svg')

## [IN PREP] Response-response correlations

In [ ]:
from scipy.signal import correlate
from scipy.stats import pearsonr

In [ ]:
r_vals = []
p_vals = []
# loop through the active averages and compute Pearson's r and p-values
# between active and passive conditions for each subject
for sx in range(len(active_avgs)):
    on_off = active_avgs[sx].time_as_index([0.050, 0.200])

    res = spearmanr(active_avgs[sx].data[0, on_off[0]:on_off[1]],
                passive_avgs[sx].data[0, on_off[0]:on_off[1]],)
    r_vals.append(res[0])
    p_vals.append(res[1])

In [ ]:
sns.histplot(r_vals, kde=True)
plt.xlabel("Pearson's correlation r")
plt.title('Correlation between active and passive conditions')
print(f"Mean r = {np.mean(r_vals):.3f}, SD = {np.std(r_vals):.3f}")

### Are active and passive correlated?

In [ ]:
from scipy.stats import ttest_1samp
from numpy import arctanh

z_vals = arctanh(r_vals)  # Fisher z-transform
t_stat, p_val = ttest_1samp(z_vals, 0)  # Null: mean r = 0

print(f"t = {t_stat:.2f}, p = {p_val:.5f}")


### Equivalence testing (for evidence of no meaningful difference)

In [ ]:
from statsmodels.stats.weightstats import DescrStatsW, ttost_paired

# Get the difference scores per timepoint, then average across time
active_data = [active_avgs[sx].data[0, on_off[0]:on_off[1]] for sx in range(len(active_avgs))]
passive_data = [passive_avgs[sx].data[0, on_off[0]:on_off[1]] for sx in range(len(active_avgs))]


active_means = np.array(active_data).mean(axis=1)
passive_means = np.array(passive_data).mean(axis=1)

# TOST: are diffs within a negligible effect size (e.g., ±0.1)?
low_eq_bound = -0.1
high_eq_bound = 0.1
result = ttost_paired(active_means, passive_means, low_eq_bound, high_eq_bound)
#print(f"TOST p-values: {res_lower:.3e}, {res_upper:.3e}")


In [ ]:
# https://www.statsmodels.org/dev/generated/statsmodels.stats.weightstats.ttost_paired.html
result

## [IN PREP] Cross-correlations 

In [ ]:
corr_array = correlate(active_avgs[0].data[0],
                        passive_avgs[0].data[0],
                        mode='full', method='fft'
                        )
plt.plot(corr_array)

lags = np.arange(-len(active_avgs[0].data[0])+1, len(active_avgs[0].data[0]))
best_lag = lags[np.argmax(corr_array)]
print(f"Best lag: {best_lag} samples ({best_lag / 1000:.3f} seconds)")


## [IN PREP] Stimulus–response correlations


In [ ]:
data.pick(['Erg1'])

In [ ]:
# epoch data based on stimulus events
stim_epochs = mne.Epochs(data_filtered, 
                    events, 
                    event_id=event_dict,
                    picks=['Erg1'],
                    tmin=-0.04, tmax=0.3, 
                    baseline=[-0.04, 0],
                    #reject = dict(eeg = 35e-6)).drop_bad()
                    reject=dict(eeg=75)).drop_bad()

In [ ]:
stim_epochs

In [ ]:
stim_epochs.average()

In [ ]:
stim_epochs.average().plot();

In [ ]:
stim_data = stim_epochs.average().get_data()[0,:]
active_data = active_grandavg.data[0]
passive_data = passive_grandavg.data[0]

In [ ]:
np.array(range(len(stim_data)))

In [ ]:
plt.plot(stim_data/10, color='grey')
plt.plot(passive_data)
plt.plot(active_data)
plt.legend(['stimulus','passive','active'])
#plt.xaxis(np.array(range(len(stim_data)))/16384)
plt.show()

In [ ]:
from scipy.stats import spearmanr
active_res = spearmanr(active_data, stim_data, alternative='greater')
passive_res = spearmanr(passive_data, stim_data, alternative='greater')

print(f"active r: {active_res.correlation:.3f}, p-value: {active_res.pvalue:.5f}")
print(f"passive r: {passive_res.correlation:.3f}, p-value: {passive_res.pvalue:.5f}")